In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('../data/cleaned_superstore.csv')

# ── Build Zone Summary ───────────────────────────────
zone_df = df.groupby(['Category', 'Sub-Category']).agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Profit  = ('Profit', 'sum'),
    Avg_Margin    = ('Profit_Margin', 'mean'),
    Avg_Discount  = ('Discount', 'mean'),
    Total_Qty     = ('Quantity', 'sum'),
    Orders        = ('Sales', 'count'),
    Loss_Orders   = ('Is_Loss', 'sum')
).reset_index()

zone_df['Loss_Rate'] = round(
    (zone_df['Loss_Orders'] / zone_df['Orders']) * 100, 1
)

In [3]:
# ── Dead Zone Scoring ────────────────────────────────
# Higher score = BETTER performance
# Lower score = DEAD ZONE

scaler = MinMaxScaler()

# Normalize positive metrics (higher is better)
zone_df[['Sales_Score', 'Profit_Score', 'Margin_Score', 'Qty_Score']] = \
    scaler.fit_transform(
        zone_df[['Total_Sales','Total_Profit','Avg_Margin','Total_Qty']]
    )

# Normalize negative metrics (higher discount = worse)
zone_df['Discount_Penalty'] = scaler.fit_transform(
    zone_df[['Avg_Discount']]
)

# Final Dead Zone Score (0 = worst dead zone, 1 = best performer)
zone_df['Performance_Score'] = round(
    (zone_df['Sales_Score']   * 0.25 +
     zone_df['Profit_Score']  * 0.35 +   # profit weighted highest
     zone_df['Margin_Score']  * 0.25 +
     zone_df['Qty_Score']     * 0.15) -
     zone_df['Discount_Penalty'] * 0.1,   # penalize high discounts
    3
)

# Tag each zone
def tag_zone(score, profit):
    if profit < 0:
        return '💀 Critical Dead Zone'
    elif score < 0.2:
        return '🔴 Dead Zone'
    elif score < 0.45:
        return '🟡 Underperforming'
    elif score < 0.7:
        return '🟢 Average'
    else:
        return '⭐ Top Performer'

zone_df['Zone_Tag'] = zone_df.apply(
    lambda row: tag_zone(row['Performance_Score'], row['Total_Profit']), axis=1
)

# Sort by score
zone_df = zone_df.sort_values('Performance_Score')

print("🎯 DEAD ZONE DETECTION RESULTS:")
print("=" * 60)
display(zone_df[['Sub-Category','Category','Total_Sales',
                  'Total_Profit','Avg_Margin','Loss_Rate',
                  'Performance_Score','Zone_Tag']])

# Save results
zone_df.to_csv('../data/zone_scores.csv', index=False)
print("\n✅ Zone scores saved!")

🎯 DEAD ZONE DETECTION RESULTS:


,Sub-Category,Category,Total_Sales,Total_Profit,Avg_Margin,Loss_Rate,Performance_Score,Zone_Tag
3,Tables,Furniture,206964.5320,-17725.4811,-14.771755,63.6,0.139,💀 Critical Dead Zone
0,Bookcases,Furniture,114879.9963,-3472.5560,-12.663904,47.8,0.152,💀 Critical Dead Zone
15,Machines,Technology,189238.6310,3384.7569,-7.202435,38.3,0.221,🟡 Underperforming
12,Supplies,Office Supplies,46673.5380,-1189.0995,11.203947,17.4,0.244,💀 Critical Dead Zone
4,Appliances,Office Supplies,107532.1610,18138.0054,-15.686910,14.4,0.275,🟡 Underperforming
8,Fasteners,Office Supplies,3024.2800,949.5182,29.917051,5.5,0.301,🟡 Underperforming
5,Art,Office Supplies,27118.7920,6527.7870,25.164573,0.0,0.384,🟡 Underperforming
7,Envelopes,Office Supplies,16476.4020,6964.1767,42.313976,0.0,0.389,🟡 Underperforming
9,Labels,Office Supplies,12486.3120,5546.2540,42.966346,0.0,0.399,🟡 Underperforming
2,Furnishings,Furniture,91705.1640,13059.1436,13.706635,17.5,0.413,🟡 Underperforming



✅ Zone scores saved!
